In [ ]:
import requests
import pandas as pd
import os
from datetime import datetime, timedelta, timezone
from pyspark.sql import SparkSession

In [ ]:
# Credenciais OAuth2

AUTH_URL = "https://login.microsoftonline.com/5e3978b9-4950-437c-87b6-492f1268dec8/oauth2/v2.0/token"
CLIENT_ID = CLIENT_ID
CLIENT_SECRET = CLIENT_SECRET
SCOPE = "https://analysis.windows.net/powerbi/api/.default"

In [ ]:
# Função para obter o token de acesso
def get_bearer_token():
    data = {
        "grant_type": "client_credentials",
        "scope": SCOPE,
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET
    }
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    response = requests.post(AUTH_URL, data=data, headers=headers)
    response.raise_for_status()
    return response.json()["access_token"]

In [ ]:
# Função para obter eventos de atividade (Admin) com continuation token
def get_activity_events(token, start_date, end_date):
    all_events = []
    continuation_token = None
    
    while True:
        if continuation_token:
            api_url = f"https://api.powerbi.com/v1.0/myorg/admin/activityevents?continuationToken={continuation_token}"
        else:
            api_url = f"https://api.powerbi.com/v1.0/myorg/admin/activityevents?startDateTime='{start_date}T00:00:00Z'&endDateTime='{end_date}T23:59:59Z'"
        
        headers = {"Authorization": f"Bearer {token}"}
        response = requests.get(api_url, headers=headers)
        response.raise_for_status()
        data = response.json()
        
        all_events.extend(data.get("activityEventEntities", []))
        continuation_token = data.get("continuationToken")
        
        if not continuation_token:
            break
    
    return all_events

In [ ]:
# Obtém o token
token = get_bearer_token()

In [ ]:
#loop de atualização do dataset
primeira_execucao = False

if primeira_execucao:
    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=27)
    mode = "overwrite"
else:
    end_date = datetime.now(timezone.utc) - timedelta(days=1)
    start_date = end_date
    mode = "append"

all_events = []
current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    events = get_activity_events(token, date_str, date_str)
    all_events.extend(events)
    current_date += timedelta(days=1)

In [ ]:
# Criação do DataFrame com limpeza preliminar
events_df = pd.DataFrame(all_events)
colunas_mantidas = ["CreationTime", "Operation", "UserId"]
events_df = events_df[colunas_mantidas]
events_df.sort_values("CreationTime", ascending=True, inplace=True)

In [ ]:
# Simples visualização da tabela resultante
display(events_df)

In [ ]:
# Salvar o arquivo em formato parquet na lakehouse do Microsoft Fabric
output_path = os.path.join("/lakehouse", "default", "Files", "Parquet")
file_path = os.path.join(output_path, "powerbi_activity_events.parquet")
events_df.to_parquet(file_path, index=False, engine='pyarrow')

print(f"Arquivo Parquet atualizado em: {file_path}")

In [ ]:
# Criar sessão Spark para trasformação do parquet em tabela delta
spark = SparkSession.builder.appName("tabela_delta").getOrCreate()


In [ ]:
# Ler o Parquet atualizado
df = spark.read.parquet(file_path)

# Adicionar os dados à tabela Delta sem sobrescrever
df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("Atividades")

print("Dados adicionados à tabela Delta com sucesso!")